# Biohub Cell Tracking — Graph-Aware Consensus Ensemble

Public "automated ensemble" notebooks rank/mean-blend a single tabular
prediction column. **That does not apply here**: each submission is a *tracking
graph* (node rows + edge rows), so there is no column to average. This notebook
ensembles the graphs directly.

**Method**
1. Cluster detections across submissions in **physical micrometres**, per
   `(dataset, timepoint)` — the same space the metric matches in
   (`z=1.625`, `y=x=0.40625` µm/voxel, 7 µm gate).
2. Keep a **consensus node** when its weighted support crosses a threshold, or
   when it belongs to a designated *anchor* submission (a trusted skeleton such
   as your best LB entry).
3. **Vote edges**: every submission that links the same consensus node pair adds
   its weight; keep edges above a threshold.
4. **Select edges** greedily by vote weight under lineage constraints
   (in-degree ≤ 1, out-degree ≤ 2; the 2nd child = a division, gated harder).
5. **Prune isolated nodes** to control the node-count over-prediction penalty.

Runs on CPU only — no GPU, no internet, no external artifacts.

> ⚠️ Ensembling only helps when the sources are **diverse**. Near-identical
> variants (e.g. LB893 with/without safe divisions) will barely move; feed in
> methodologically different models (classical DoG/Hungarian, learned U-Net/ILP,
> LB893) for real gains. The diagnostics below report pairwise agreement so you
> can see how much diversity you actually have.


## 1. Configuration

Point `SOURCES` at each submission's `submission.csv`. On Kaggle these come from
attached datasets or other notebooks' outputs
(`/kaggle/input/<dataset>/submission.csv` or
`/kaggle/input/notebooks/<owner>/<slug>/submission.csv`). Give the strongest
entry the highest `weight` and optionally mark it as the `anchor`.


In [ ]:
# ── Sources to ensemble ───────────────────────────────────────────────────────
# Each entry: name, path to submission.csv, weight (>0). Higher weight = more
# trusted. Set exactly one "anchor": True to use it as a protected skeleton.
SOURCES = [
    # Pre-filled with the three most *diverse* candidates. Attach each kernel's
    # output under Add Data > Notebook Output; private outputs mount at
    # /kaggle/input/notebooks/<owner>/<slug>/submission.csv (verify the paths).
    {"name": "lb893-0.893", "weight": 3.0, "anchor": True,
     "path": "/kaggle/input/notebooks/dalloliogm/lb893-learned-graph-tracker-micro-safe-divisi/submission.csv"},
    {"name": "classical-nms-0.834", "weight": 1.0,
     "path": "/kaggle/input/notebooks/dalloliogm/biohub-nms-3-8-submission-candidate/submission.csv"},
    {"name": "learned-ilp-0.810", "weight": 1.0,
     "path": "/kaggle/input/notebooks/dalloliogm/biohub-learned-unet-ilp-candidate/submission.csv"},
]

# ── Consensus thresholds (fractions of total source weight) ───────────────────
NODE_MERGE_UM  = 3.0    # detections within this physical distance = same cell
TAU_NODE_FRAC  = 0.50   # keep a node with >= this fraction of weighted support
TAU_EDGE_FRAC  = 0.50   # keep an edge with >= this fraction of weighted votes
TAU_DIV_FRAC   = 0.80   # a 2nd outgoing edge (division) needs this much support
KEEP_ANCHOR_NODES = True   # always keep the anchor's nodes
KEEP_ANCHOR_EDGES = False  # always keep the anchor's edges (use with care)
PRUNE_ISOLATED = True   # drop consensus nodes left with no edges

# ── Validation (optional) ─────────────────────────────────────────────────────
# When True, ensemble per-source predictions on labeled TRAIN movies and score
# with the official metric. Requires the support pack + each source's train-movie
# submission. Leave False to just produce a test submission.
VALIDATION_MODE = False
TRAIN_SOURCES = []      # same shape as SOURCES but pointing at train-movie CSVs
SUPPORT_PACK_REPO = "/kaggle/input/biohub-tracking-support-pack-50ep-v1"
TRAIN_DIR = "/kaggle/input/biohub-cell-tracking-during-development/train"

OUTPUT_PATH = "/kaggle/working/submission.csv"

# ── Auto-discover public base models (optional) ───────────────────────────────
# When True, scan the competition's top public notebooks, use an LLM to keep only
# ORIGINAL detection+tracking pipelines (dropping ensembles/blends-of-blends),
# download their submission outputs, de-duplicate, and use them as SOURCES.
# REQUIRES: notebook internet ON + Kaggle Secrets (KAGGLE_USERNAME, KAGGLE_KEY,
# and OPENROUTER_API_KEY if LLM_FILTER=True). Leave False for an internet-off
# submission run that just uses the SOURCES listed above.
DISCOVER_PUBLIC = False
DISCOVER_TOP_N  = 25          # how many top-scored public notebooks to consider
DISCOVER_BEST   = "high"      # this metric: higher is better
LLM_FILTER      = True        # LLM base-model vs blend classifier
LLM_MODEL       = "google/gemini-2.5-flash"
EXCLUDE_BLENDS_BY_NAME = True  # cheap keyword prefilter before the LLM
DEDUP_AGREEMENT = 0.98        # drop a submission that agrees this much with a better one
ANCHOR_WEIGHT   = 3.0         # weight for the top-ranked discovered base model


## 2. Consensus core

(Physical-space node clustering + weighted edge voting, with lineage constraints.)

In [ ]:
"""Graph-aware consensus ensembler for the Biohub cell-tracking competition.

Submissions are tracking graphs (node rows + edge rows), not tabular prediction
columns, so this cannot use rank/mean blending. Instead we:

  1. Cluster nodes across submissions in *physical* micrometres, per (dataset, t).
  2. Keep a consensus node when its weighted support crosses a threshold
     (or it belongs to the anchor submission, if one is designated).
  3. Vote edges: a consensus edge accumulates the weight of every submission
     whose own edge maps onto the same consensus node pair.
  4. Select edges greedily by vote weight under lineage constraints
     (in-degree <= 1, out-degree <= 2, second child = division gated harder).
  5. Prune isolated consensus nodes to control the node-count penalty.

Depends only on numpy / pandas / scipy so it runs anywhere (no GPU/artifacts).
"""
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

# Physical voxel scale (micrometres per voxel) from the competition metric.
VOXEL_SCALE_UM = np.array([1.625, 0.40625, 0.40625])  # (z, y, x)

SUBMISSION_COLUMNS = [
    "id", "dataset", "row_type", "node_id", "t", "z", "y", "x",
    "source_id", "target_id",
]


@dataclass
class ConsensusConfig:
    node_merge_um: float = 3.0       # radius to treat detections as the same cell
    tau_node_frac: float = 0.5       # keep node if weighted support >= frac * total_weight
    tau_edge_frac: float = 0.5       # keep edge if weighted votes  >= frac * total_weight
    tau_div_frac: float = 0.8        # 2nd outgoing edge needs this much weighted support
    anchor_idx: int | None = None    # index of a "trusted skeleton" submission
    keep_anchor_nodes: bool = True   # always retain the anchor's nodes
    keep_anchor_edges: bool = False  # always retain the anchor's edges
    prune_isolated: bool = True      # drop consensus nodes with no kept edges


@dataclass
class ConsensusStats:
    per_dataset: list[dict] = field(default_factory=list)

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.per_dataset)


def load_submission(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing = [c for c in SUBMISSION_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"{path}: missing columns {missing}")
    for c in ["node_id", "t", "z", "y", "x", "source_id", "target_id"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["row_type"] = df["row_type"].astype(str)
    df["dataset"] = df["dataset"].astype(str)
    return df


def _cluster_frame_nodes(
    coords_um: np.ndarray,
    src_idx: np.ndarray,
    weights: np.ndarray,
    order: np.ndarray,
    merge_um: float,
):
    """Greedy seed clustering with <=1 node per source per cluster.

    `order` lists point indices in the priority they should seed clusters
    (anchor first, then by descending weight). Returns cluster assignment array
    (cluster id per point, -1 for none) and list of member-index lists.
    """
    from scipy.spatial import cKDTree

    n = coords_um.shape[0]
    assigned = np.full(n, -1, dtype=np.int64)
    clusters: list[list[int]] = []
    if n == 0:
        return assigned, clusters
    tree = cKDTree(coords_um)

    for seed in order:
        if assigned[seed] != -1:
            continue
        cid = len(clusters)
        neigh = tree.query_ball_point(coords_um[seed], merge_um)
        # nearest unassigned point per source (seed's own source included)
        best_per_source: dict[int, tuple[float, int]] = {}
        for j in neigh:
            if assigned[j] != -1:
                continue
            s = int(src_idx[j])
            d = float(np.linalg.norm(coords_um[j] - coords_um[seed]))
            if s not in best_per_source or d < best_per_source[s][0]:
                best_per_source[s] = (d, j)
        members = [j for _, j in best_per_source.values()]
        for j in members:
            assigned[j] = cid
        clusters.append(members)
    return assigned, clusters


def build_consensus(
    sources: list[pd.DataFrame],
    weights: list[float],
    config: ConsensusConfig,
) -> tuple[pd.DataFrame, ConsensusStats]:
    assert len(sources) == len(weights) and sources, "need >=1 weighted source"
    weights = [float(w) for w in weights]
    total_w = sum(weights)
    stats = ConsensusStats()

    datasets = sorted({d for df in sources for d in df["dataset"].unique()})
    out_node_rows: list[dict] = []
    out_edge_rows: list[dict] = []
    next_node_id = 1  # globally unique node ids

    for dataset in datasets:
        # Per-source node tables for this dataset (indexed by original node_id).
        node_tabs = []
        edge_tabs = []
        for df in sources:
            g = df[df["dataset"] == dataset]
            node_tabs.append(g[g["row_type"] == "node"])
            edge_tabs.append(g[g["row_type"] == "edge"])

        # (source_idx, node_id) -> consensus node id (only for KEPT clusters)
        node_map: dict[tuple[int, int], int] = {}
        # consensus id -> (t, z, y, x voxels)
        consensus_nodes: dict[int, tuple[int, float, float, float]] = {}

        timepoints = sorted({
            int(t) for tab in node_tabs for t in tab["t"].dropna().unique()
        })
        for t in timepoints:
            pts, src_idx, weight_arr, seed_pri, ref = [], [], [], [], []
            for si, tab in enumerate(node_tabs):
                sub = tab[tab["t"] == t]
                for row in sub.itertuples(index=False):
                    z, y, x = float(row.z), float(row.y), float(row.x)
                    pts.append([z, y, x])
                    src_idx.append(si)
                    weight_arr.append(weights[si])
                    # anchor seeds first, then heavier sources first
                    is_anchor = (config.anchor_idx == si)
                    seed_pri.append((0 if is_anchor else 1, -weights[si]))
                    ref.append((si, int(row.node_id), z, y, x))
            if not pts:
                continue
            pts = np.asarray(pts, dtype=np.float64)
            coords_um = pts * VOXEL_SCALE_UM
            src_idx = np.asarray(src_idx)
            weight_arr = np.asarray(weight_arr)
            order = np.array(sorted(range(len(pts)), key=lambda j: seed_pri[j]))

            assigned, clusters = _cluster_frame_nodes(
                coords_um, src_idx, weight_arr, order, config.node_merge_um
            )

            for cid, members in enumerate(clusters):
                m_src = {int(src_idx[j]) for j in members}
                support = sum(weights[s] for s in m_src)
                has_anchor = config.anchor_idx in m_src
                keep = support >= config.tau_node_frac * total_w
                if config.keep_anchor_nodes and has_anchor:
                    keep = True
                if not keep:
                    continue
                w_members = np.array([weight_arr[j] for j in members])
                cen = np.average(pts[members], axis=0, weights=w_members)
                nid = next_node_id
                next_node_id += 1
                consensus_nodes[nid] = (
                    t, int(round(cen[0])), int(round(cen[1])), int(round(cen[2]))
                )
                for j in members:
                    si, onid = ref[j][0], ref[j][1]
                    node_map[(si, onid)] = nid

        # ---- edge voting ----
        votes: dict[tuple[int, int], float] = defaultdict(float)
        anchor_edges: set[tuple[int, int]] = set()
        for si, etab in enumerate(edge_tabs):
            for row in etab.itertuples(index=False):
                cu = node_map.get((si, int(row.source_id)))
                cv = node_map.get((si, int(row.target_id)))
                if cu is None or cv is None:
                    continue
                votes[(cu, cv)] += weights[si]
                if config.anchor_idx == si:
                    anchor_edges.add((cu, cv))

        candidates = []
        for (cu, cv), w in votes.items():
            keep = w >= config.tau_edge_frac * total_w
            if config.keep_anchor_edges and (cu, cv) in anchor_edges:
                keep = True
            if keep:
                candidates.append((w, cu, cv))
        candidates.sort(key=lambda r: (-r[0], r[1], r[2]))

        in_deg: dict[int, int] = defaultdict(int)
        out_deg: dict[int, int] = defaultdict(int)
        kept_edges: list[tuple[int, int]] = []
        for w, cu, cv in candidates:
            t_u = consensus_nodes[cu][0]
            t_v = consensus_nodes[cv][0]
            if t_v != t_u + 1:
                continue  # must be strictly consecutive
            if in_deg[cv] >= 1:
                continue
            if out_deg[cu] >= 2:
                continue
            if out_deg[cu] == 1 and w < config.tau_div_frac * total_w:
                continue  # division: harder gate for the 2nd child
            kept_edges.append((cu, cv))
            in_deg[cv] += 1
            out_deg[cu] += 1

        # ---- prune isolated consensus nodes ----
        used = set()
        for cu, cv in kept_edges:
            used.add(cu)
            used.add(cv)
        keep_nodes = set(consensus_nodes) if not config.prune_isolated else used

        n_nodes = 0
        n_div = 0
        for nid in sorted(keep_nodes):
            t, z, y, x = consensus_nodes[nid]
            out_node_rows.append({
                "dataset": dataset, "row_type": "node", "node_id": nid,
                "t": t, "z": z, "y": y, "x": x,
                "source_id": -1, "target_id": -1,
            })
            n_nodes += 1
        for cu, cv in kept_edges:
            out_edge_rows.append({
                "dataset": dataset, "row_type": "edge", "node_id": -1,
                "t": -1, "z": -1, "y": -1, "x": -1,
                "source_id": cu, "target_id": cv,
            })
        n_div = sum(1 for c in out_deg.values() if c >= 2)
        stats.per_dataset.append({
            "dataset": dataset, "nodes": n_nodes, "edges": len(kept_edges),
            "divisions": n_div,
        })

    body_cols = [c for c in SUBMISSION_COLUMNS if c != "id"]
    out = pd.DataFrame(out_node_rows + out_edge_rows, columns=body_cols)
    out.insert(0, "id", range(len(out)))
    out = out[SUBMISSION_COLUMNS]
    for c in ["node_id", "t", "z", "y", "x", "source_id", "target_id"]:
        out[c] = out[c].astype(int)
    return out.reset_index(drop=True), stats


def node_agreement(a: pd.DataFrame, b: pd.DataFrame, merge_um: float = 3.0) -> float:
    """Fraction of `a`'s nodes within `merge_um` of a `b` node in the same
    (dataset, t). ~1.0 => the two submissions are near-duplicates."""
    from scipy.spatial import cKDTree

    na = a[a["row_type"] == "node"]
    nb = b[b["row_type"] == "node"]
    total = matched = 0
    for d in sorted(set(na["dataset"]) & set(nb["dataset"])):
        ga = na[na["dataset"] == d]
        gb = nb[nb["dataset"] == d]
        gb_by_t = {int(t): g for t, g in gb.groupby("t")}
        for t, ca in ga.groupby("t"):
            total += len(ca)
            cb = gb_by_t.get(int(t))
            if cb is None or len(cb) == 0:
                continue
            tree = cKDTree(cb[["z", "y", "x"]].to_numpy() * VOXEL_SCALE_UM)
            dd, _ = tree.query(ca[["z", "y", "x"]].to_numpy() * VOXEL_SCALE_UM, k=1)
            matched += int((dd <= merge_um).sum())
    return matched / total if total else float("nan")


def dedup_by_agreement(
    dfs: list[pd.DataFrame],
    names: list[str],
    scores: list[float],
    threshold: float = 0.98,
    merge_um: float = 3.0,
) -> list[int]:
    """Drop near-duplicate submissions. When two submissions agree above
    `threshold` (symmetric), keep the higher-scored one. Returns kept indices."""
    order = sorted(range(len(dfs)), key=lambda i: -scores[i])  # best first
    kept: list[int] = []
    for i in order:
        dup_of = None
        for j in kept:
            ag = 0.5 * (node_agreement(dfs[i], dfs[j], merge_um)
                        + node_agreement(dfs[j], dfs[i], merge_um))
            if ag >= threshold:
                dup_of = j
                break
        if dup_of is None:
            kept.append(i)
        else:
            print(f"  dedup: '{names[i]}' ~= '{names[dup_of]}' "
                  f"(agree>={threshold}); dropping lower score")
    return sorted(kept)


def structural_check(df: pd.DataFrame) -> dict:
    nodes = df[df["row_type"] == "node"]
    edges = df[df["row_type"] == "edge"]
    node_ids = set(nodes["node_id"].tolist())
    assert nodes["node_id"].is_unique, "duplicate node_id"
    problems = []
    # endpoints exist
    src_ok = edges["source_id"].isin(node_ids).all()
    tgt_ok = edges["target_id"].isin(node_ids).all()
    if not src_ok or not tgt_ok:
        problems.append("edge endpoint missing from nodes")
    # consecutive frames + degree limits
    tmap = dict(zip(nodes["node_id"], nodes["t"]))
    in_deg: dict[int, int] = defaultdict(int)
    out_deg: dict[int, int] = defaultdict(int)
    for s, tt in zip(edges["source_id"], edges["target_id"]):
        if tmap.get(tt) != tmap.get(s, -99) + 1:
            problems.append(f"nonconsecutive edge {s}->{tt}")
        in_deg[tt] += 1
        out_deg[s] += 1
    max_in = max(in_deg.values(), default=0)
    max_out = max(out_deg.values(), default=0)
    if max_in > 1:
        problems.append(f"max indegree {max_in} > 1")
    if max_out > 2:
        problems.append(f"max outdegree {max_out} > 2")
    return {
        "rows": len(df), "nodes": len(nodes), "edges": len(edges),
        "max_indegree": max_in, "max_outdegree": max_out,
        "divisions": sum(1 for v in out_deg.values() if v >= 2),
        "problems": problems[:10],
        "ok": not problems,
    }


## 2b. (Optional) Auto-discover public base models

When `DISCOVER_PUBLIC = True`, this scans the competition's top-scored public
notebooks and uses an LLM to keep only **original** detection+tracking pipelines,
dropping ensembles / blends-of-blends (a blend of blends adds no new signal and
double-counts models already in the pool). It then downloads each kept notebook's
`submission.csv`, validates the tracking schema, de-duplicates near-identical
graphs, and overwrites `SOURCES` — weighting by the score parsed from the title
and making the top scorer the anchor.

**Requires notebook internet ON + Kaggle Secrets** (`KAGGLE_USERNAME`,
`KAGGLE_KEY`, and `OPENROUTER_API_KEY` if `LLM_FILTER=True`). This is the
*discovery* phase; for a final internet-off submission run, do discovery once,
save the pool, then re-run with `DISCOVER_PUBLIC=False`.

> ⚠️ Note on rules: this blends other people's **public** submissions. That is a
> common Kaggle practice and permitted under public code sharing, but it will not
> teach you anything your own models don't already know — treat it as a
> leaderboard tool, and keep an original model of your own in the mix.


In [ ]:
def _load_kaggle_secrets():
    import os
    try:
        from kaggle_secrets import UserSecretsClient
        s = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = s.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = s.get_secret("KAGGLE_KEY")
        if LLM_FILTER:
            try:
                os.environ["OPENROUTER_API_KEY"] = s.get_secret("OPENROUTER_API_KEY")
            except Exception as e:
                print(f"No OPENROUTER_API_KEY ({e}); LLM filter will be off.")
        return True
    except Exception as e:
        print(f"Could not load Kaggle Secrets: {e}")
        return False


def get_kernels_list(competition, page_size=75, best="high"):
    import subprocess
    sort_by = "scoreDescending" if best == "high" else "scoreAscending"
    cmd = ["kaggle", "kernels", "list", "--competition", competition,
           "--page-size", str(page_size), "--csv", "--sort-by", sort_by, "-v"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("kernels list failed:", r.stderr[:200]); return []
    lines = [l for l in r.stdout.strip().split("\n")
             if l and not l.startswith(("Warning:", "Info:"))]
    if len(lines) < 2:
        return []
    headers = lines[0].split(",")
    out = []
    for line in lines[1:]:
        vals = line.split(",")
        if len(vals) >= len(headers):
            out.append(dict(zip(headers, vals)))
    return out


def get_notebook_source(kernel_ref):
    import subprocess, tempfile, json as _json
    from pathlib import Path
    with tempfile.TemporaryDirectory() as tmp:
        r = subprocess.run(["kaggle", "kernels", "pull", kernel_ref, "-p", tmp],
                           capture_output=True, text=True)
        if r.returncode != 0:
            return ""
        for f in Path(tmp).iterdir():
            if f.suffix == ".ipynb":
                nb = _json.loads(f.read_text(encoding="utf-8"))
                return "\n\n".join("".join(c.get("source", [])) for c in nb.get("cells", []))
            if f.suffix == ".py":
                return f.read_text(encoding="utf-8")
    return ""


def download_kernel_output(kernel_ref, out_dir):
    import subprocess
    from pathlib import Path
    r = subprocess.run(["kaggle", "kernels", "output", kernel_ref, "-p", str(out_dir)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        return None
    csvs = [f for f in Path(out_dir).iterdir() if f.suffix == ".csv"]
    for f in csvs:
        if f.name == "submission.csv":
            return f
    return csvs[0] if csvs else None


def extract_score_from_title(title):
    """Best-effort LB score HINT from a title (display only; titles are not
    trustworthy). Handles 0.893, LB893, lb839, lb-0-73, LB 0.835."""
    import re
    t = title.lower()
    m = re.search(r"([01]\.[0-9]{2,4})", t)                 # 0.893
    if m:
        return float(m.group(1))
    m = re.search(r"lb[\s_\-]*0[\s_\-]*([0-9]{2,3})\b", t)  # lb-0-73 -> 0.73
    if m:
        return float("0." + m.group(1))
    m = re.search(r"lb[\s_\-]*([0-9]{2,3})\b", t)            # lb893 -> 0.893
    if m:
        return float("0." + m.group(1))
    return None


BLEND_KEYWORDS = ["blend", "ensemble", "stack", "mix", "combine", "merge",
                  "avg", "average", "mean", "median", "weighted", "meta",
                  "voting", "consensus", "wisdom"]


def is_blend_by_name(title):
    t = title.lower()
    return any(k in t for k in BLEND_KEYWORDS)


def parse_llm_json_response(text):
    import json as _json, re
    text = text.strip()
    try:
        return _json.loads(text)
    except _json.JSONDecodeError:
        pass
    m = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if m:
        try:
            return _json.loads(m.group(1).strip())
        except _json.JSONDecodeError:
            pass
    m = re.search(r"\{[^{}]*\"is_blend\"[^{}]*\}", text, re.DOTALL)
    if m:
        try:
            return _json.loads(m.group(0))
        except _json.JSONDecodeError:
            pass
    return {"is_blend": False, "confidence": "low", "reason": "unparseable"}


def analyze_is_blend(source, api_key, model):
    """LLM classifies a cell-tracking notebook as a blend/ensemble of other
    submissions vs. an original detection+tracking pipeline."""
    import requests
    src = source[:15000] + ("\n...[truncated]..." if len(source) > 15000 else "")
    prompt = f"""You are analyzing a Kaggle notebook for a 3D+time CELL TRACKING
competition. Submissions are tracking GRAPHS (node rows = cell detections, edge
rows = links across frames), not a single prediction column.

Decide if this notebook is a BLEND/ENSEMBLE of other people's submissions, or an
ORIGINAL tracking pipeline.

A BLEND/ENSEMBLE notebook:
- loads OTHER submission.csv files (other kernels' outputs) and merges/votes/
  averages them, or picks per-dataset among them
- does NOT run its own detector or linker on the image data
- mentions blend, ensemble, consensus, voting, merge of submissions

An ORIGINAL notebook:
- runs its OWN cell detection (e.g. DoG, U-Net) and its OWN linking/tracking
  (Hungarian, ILP, motion model) on the competition images/zarr
- may ensemble ITS OWN models internally (still original)

NOTEBOOK CONTENT:
{src}

Respond with ONLY a JSON object (no markdown):
{{"is_blend": true/false, "confidence": "high/medium/low", "reason": "brief"}}"""
    data = {"model": model, "temperature": 0.1, "max_tokens": 200,
            "messages": [{"role": "user", "content": prompt}]}
    r = requests.post("https://openrouter.ai/api/v1/chat/completions",
                      headers={"Authorization": f"Bearer {api_key}",
                               "Content-Type": "application/json"},
                      json=data, timeout=60)
    r.raise_for_status()
    return parse_llm_json_response(r.json()["choices"][0]["message"]["content"])


def is_valid_tracking_submission(df):
    try:
        need = set(SUBMISSION_COLUMNS)
        if not need.issubset(df.columns):
            return False
        return (df["row_type"] == "node").any() and (df["row_type"] == "edge").any()
    except Exception:
        return False


In [ ]:
if DISCOVER_PUBLIC:
    import os, shutil
    from pathlib import Path

    assert _load_kaggle_secrets(), "Kaggle Secrets required for discovery"
    api_key = os.getenv("OPENROUTER_API_KEY") if LLM_FILTER else None
    use_llm = bool(LLM_FILTER and api_key)
    if LLM_FILTER and not use_llm:
        print("LLM_FILTER on but no OPENROUTER_API_KEY -> name-only prefilter.")

    pool_dir = Path("/kaggle/working/discovered")
    if pool_dir.exists():
        shutil.rmtree(pool_dir)
    pool_dir.mkdir(parents=True)

    kernels = get_kernels_list(COMPETITION, page_size=DISCOVER_TOP_N * 3, best=DISCOVER_BEST)
    print(f"Scanning {len(kernels)} public notebooks for {COMPETITION}\n")
    print(f"{'#':<3}{'notebook':<44}{'score':<8}{'status'}")
    print("-" * 78)

    discovered, kept = [], 0
    for k in kernels:
        if kept >= DISCOVER_TOP_N:
            break
        ref = k.get("ref", "")
        title = k.get("title", ref)
        name = ref.split("/")[-1]
        score = extract_score_from_title(title)
        sc = f"{score:.3f}" if score is not None else "-"

        if EXCLUDE_BLENDS_BY_NAME and is_blend_by_name(title):
            print(f"   {name[:42]:<42}{sc:<8}skip: blend-by-name"); continue
        if use_llm:
            src = get_notebook_source(ref)
            if src:
                try:
                    a = analyze_is_blend(src, api_key, LLM_MODEL)
                    if a.get("is_blend"):
                        print(f"   {name[:42]:<42}{sc:<8}LLM blend ({a.get('confidence')}: {a.get('reason','')[:24]})")
                        continue
                except Exception as e:
                    print(f"   {name[:42]:<42}{sc:<8}LLM err ({str(e)[:20]}); keeping")

        odir = pool_dir / f"sub_{kept+1:02d}_{name[:28]}"
        odir.mkdir(parents=True, exist_ok=True)
        csv = download_kernel_output(ref, odir)
        if csv is None:
            print(f"   {name[:42]:<42}{sc:<8}no output"); shutil.rmtree(odir, ignore_errors=True); continue
        try:
            df = load_submission(str(csv))
        except Exception as e:
            print(f"   {name[:42]:<42}{sc:<8}bad csv ({str(e)[:20]})"); shutil.rmtree(odir, ignore_errors=True); continue
        if not is_valid_tracking_submission(df):
            print(f"   {name[:42]:<42}{sc:<8}not a tracking graph"); shutil.rmtree(odir, ignore_errors=True); continue
        kept += 1
        dst = pool_dir / f"sub_{kept:02d}_{name[:28]}.csv"
        df.to_csv(dst, index=False)
        shutil.rmtree(odir, ignore_errors=True)
        discovered.append({"name": name, "path": str(dst), "score": score})
        print(f"{kept:<3}{name[:42]:<42}{sc:<8}OK")

    print("-" * 78)
    assert discovered, "Discovery kept 0 notebooks; loosen filters or check auth."

    # De-duplicate near-identical graphs, keeping the higher-scored one.
    dfs = [load_submission(d["path"]) for d in discovered]
    names = [d["name"] for d in discovered]
    scores = [d["score"] if d["score"] is not None else 0.0 for d in discovered]
    print("\nDe-duplicating discovered graphs...")
    keep_idx = dedup_by_agreement(dfs, names, scores, DEDUP_AGREEMENT, NODE_MERGE_UM)
    discovered = [discovered[i] for i in keep_idx]

    # Anchor = rank-0 kept (Kaggle already sorted best-first; dedup preserved
    # that order). Titles/claimed scores are not trustworthy, so weights are
    # uniform except the anchor, which gets ANCHOR_WEIGHT as the trusted skeleton.
    SOURCES = []
    for i, d in enumerate(discovered):
        hint = f"{d['score']:.3f}" if d["score"] is not None else "-"
        SOURCES.append({"name": d["name"], "path": d["path"],
                        "weight": ANCHOR_WEIGHT if i == 0 else 1.0,
                        "anchor": (i == 0), "score_hint": hint})
    print(f"\nDiscovery kept {len(SOURCES)} base models -> SOURCES:")
    for s in SOURCES:
        print(f"  {s['name'][:38]:<38} w={s['weight']:<5} anchor={str(s['anchor']):<5} score_hint={s['score_hint']}")
else:
    print("DISCOVER_PUBLIC = False -> using the manually listed SOURCES.")


## 3. Load sources & report diversity

Loads every source, validates the schema, and reports how much the submissions
actually disagree — near-identical inputs cannot produce an ensemble gain.


In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

assert SOURCES, "Fill in SOURCES first (paths to each submission.csv)."

source_dfs, source_weights, source_names = [], [], []
anchor_idx = None
for i, s in enumerate(SOURCES):
    df = load_submission(s["path"])
    source_dfs.append(df)
    source_weights.append(float(s.get("weight", 1.0)))
    source_names.append(s.get("name", f"src{i}"))
    if s.get("anchor"):
        assert anchor_idx is None, "only one source may be the anchor"
        anchor_idx = i

print(f"Loaded {len(source_dfs)} sources:")
for name, df, w in zip(source_names, source_dfs, source_weights):
    n = int((df.row_type == "node").sum()); e = int((df.row_type == "edge").sum())
    print(f"  {name:>16}  w={w:<4}  nodes={n:>8}  edges={e:>8}  datasets={df.dataset.nunique()}")
if anchor_idx is not None:
    print(f"Anchor: {source_names[anchor_idx]}")

# Pairwise node-agreement (from the core): fraction of source A's nodes within
# NODE_MERGE_UM of a source B node in the same (dataset, t). High => low diversity.
if len(source_dfs) > 1:
    print("\nPairwise node agreement (frac of row's nodes matched in col):")
    hdr = "".join(f"{n[:10]:>12}" for n in source_names)
    print(f"{'':>12}{hdr}")
    for i, a in enumerate(source_dfs):
        row = "".join(f"{node_agreement(a, b, NODE_MERGE_UM):>12.3f}" for b in source_dfs)
        print(f"{source_names[i][:10]:>12}{row}")


## 4. Build the consensus submission

In [ ]:
cfg = ConsensusConfig(
    node_merge_um=NODE_MERGE_UM,
    tau_node_frac=TAU_NODE_FRAC,
    tau_edge_frac=TAU_EDGE_FRAC,
    tau_div_frac=TAU_DIV_FRAC,
    anchor_idx=anchor_idx,
    keep_anchor_nodes=KEEP_ANCHOR_NODES,
    keep_anchor_edges=KEEP_ANCHOR_EDGES,
    prune_isolated=PRUNE_ISOLATED,
)
consensus_df, stats = build_consensus(source_dfs, source_weights, cfg)

chk = structural_check(consensus_df)
print("Structural check:", {k: chk[k] for k in
      ["ok", "rows", "nodes", "edges", "max_indegree", "max_outdegree", "divisions"]})
assert chk["ok"], f"Structural problems: {chk['problems']}"

print("\nPer-dataset consensus:")
display(stats.to_frame())

# Every test dataset must appear in the submission.
test_datasets = sorted({d for df in source_dfs for d in df.dataset.unique()})
missing = set(test_datasets) - set(consensus_df.dataset.unique())
assert not missing, f"Datasets missing from consensus: {missing}"

consensus_df.to_csv(OUTPUT_PATH, index=False)
print(f"\nWrote {len(consensus_df)} rows -> {OUTPUT_PATH}")
print(consensus_df.head())


## 5. Optional — validate on labeled train movies (official metric)

Only runs when `VALIDATION_MODE = True`. Ensembles each source's **train-movie**
predictions and scores the result with the competition's own
`biohub_tracking.metrics.evaluate`, so you can tune `TAU_*` on the exact metric
(embryo-disjoint) instead of a proxy. Requires the support pack and per-source
train submissions in `TRAIN_SOURCES`.


In [ ]:
if VALIDATION_MODE:
    import sys, json as _json
    from pathlib import Path
    import polars as pl
    import tracksdata as td

    sys.path.insert(0, str(Path(SUPPORT_PACK_REPO) / "src"))
    from biohub_tracking.metrics import evaluate, node_recall, per_sample_metrics, summarise
    K = td.DEFAULT_ATTR_KEYS

    def graph_from_submission_rows(dataset, table):
        g = table[table.dataset == dataset]
        nodes = g[g.row_type == "node"]; edges = g[g.row_type == "edge"]
        graph = td.graph.InMemoryGraph()
        for key in (K.Z, K.Y, K.X):
            graph.add_node_attr_key(key, pl.Float64, 0.0)
        nmap = {}
        for r in nodes.itertuples(index=False):
            nmap[int(r.node_id)] = graph.add_node(
                {K.T: int(r.t), K.Z: float(r.z), K.Y: float(r.y), K.X: float(r.x)})
        for r in edges.itertuples(index=False):
            s, t = int(r.source_id), int(r.target_id)
            if s in nmap and t in nmap:
                graph.add_edge(nmap[s], nmap[t], {})
        return graph

    def graph_from_geff(path):  # provided by support pack helpers
        from biohub_tracking.io import graph_from_geff as _g
        return _g(path)

    assert TRAIN_SOURCES, "VALIDATION_MODE needs TRAIN_SOURCES"
    train_dfs = [load_submission(s["path"]) for s in TRAIN_SOURCES]
    train_w = [float(s.get("weight", 1.0)) for s in TRAIN_SOURCES]
    train_anchor = next((i for i, s in enumerate(TRAIN_SOURCES) if s.get("anchor")), None)
    vcfg = ConsensusConfig(NODE_MERGE_UM, TAU_NODE_FRAC, TAU_EDGE_FRAC, TAU_DIV_FRAC,
                           train_anchor, KEEP_ANCHOR_NODES, KEEP_ANCHOR_EDGES, PRUNE_ISOLATED)
    ens_train, _ = build_consensus(train_dfs, train_w, vcfg)

    rows = []
    for dataset in sorted(ens_train.dataset.unique()):
        truth = Path(TRAIN_DIR) / f"{dataset}.geff"
        if not truth.exists():
            print(f"skip {dataset}: no {truth}"); continue
        pred_g = graph_from_submission_rows(dataset, ens_train)
        res = evaluate(pred_g, graph_from_geff(truth), scale=list(VOXEL_SCALE_UM), max_distance=7.0)
        rec = node_recall(pred_g, graph_from_geff(truth))
        rows.append({"dataset": dataset, **per_sample_metrics(res, float("nan"), rec)})
        print(dataset, rows[-1].get("adj_edge_jaccard"))
    print("\nSUMMARY:", _json.dumps(summarise(rows), indent=2))
else:
    print("VALIDATION_MODE = False (skipped). Set it True + fill TRAIN_SOURCES to tune thresholds.")
